# LLM-Powered Conversation Analysis

This notebook uses the OpenAI API to semantically analyse your chat history. It:

1. Groups messages into conversation-level summaries
2. Sends each conversation to an LLM for topic, use-case, and complexity classification
3. Aggregates themes per platform (ChatGPT vs Claude)
4. Compares usage patterns across the two platforms

> **Cost note:** This notebook calls the OpenAI API once per conversation (up to `sample_size`, default 30).
> At gpt-4o-mini rates this is typically < $0.10 for a typical chat history.

## Prerequisites

**1. Parse your exports first**

```bash
ocmem parse path/to/chatgpt_export.zip --output data/staging/chatgpt_messages.jsonl
ocmem parse path/to/claude_export.zip  --output data/staging/claude_messages.jsonl
```

**2. Set your OpenAI API key**

Create a `.env` file in the project root (copy from `.env.example`):

```
OPENAI_API_KEY=sk-...
```

**3. Install dependencies**

```bash
pip install pandas openai python-dotenv
```

## Step 1: Imports and Configuration

Change `DATA_DIR` and `MODEL` here if needed.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from openai import OpenAI

DATA_DIR = Path("data/staging")
OUTPUT_DIR = Path("data/figures")

MODEL = "gpt-4o-mini"

## Step 2: Helper Functions

Run all cells in this section before the analysis.

### `get_openai_client`
Reads `OPENAI_API_KEY` from the environment and returns a configured `OpenAI` client. Raises a clear error if the key is missing.

In [ ]:
def get_openai_client():
    """Initialize OpenAI client with API key check"""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable not set")
    return OpenAI(api_key=api_key)

### `load_chats`
Reads a `.jsonl` file and returns a flat pandas DataFrame — the format produced by `ocmem parse`.

In [ ]:
def load_chats(filepath: Path) -> pd.DataFrame:
    """Load JSONL chat data"""
    data = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(data)

### `get_conversation_summaries`
Aggregates the flat message table into one row per conversation, joining the first 10 user messages into a single string for LLM input.

In [ ]:
def get_conversation_summaries(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    """Aggregate messages into conversation-level summaries"""
    user_role = "user" if platform == "chatgpt" else "human"

    conversations = (
        df[df["role"] == user_role]
        .groupby("conversation_id")
        .agg(
            {
                "content": lambda x: " | ".join(x.astype(str).tolist()[:10]),
                "title": "first",
                "conversation_create_time": "first",
                "message_id": "count",
            }
        )
        .rename(columns={"message_id": "message_count", "content": "user_queries"})
        .reset_index()
    )

    conversations = conversations[conversations["message_count"] >= 3]
    conversations = conversations.sort_values("message_count", ascending=False)

    return conversations

### `analyze_conversation_with_llm`
Sends a single conversation summary to the LLM and asks it to identify topic, use-case category, complexity, and user intent.

In [ ]:
def analyze_conversation_with_llm(client, title: str, queries: str, platform: str) -> dict:
    """Analyze a single conversation using Responses API"""

    input_text = f"""Platform: {platform.upper()}
Conversation Title: {title}
User Queries (first 10): {queries[:2000]}

Analyze this conversation and identify:
1. Primary topic/domain (1-3 words)
2. Use case category (e.g., coding, writing, research, learning, debugging, design)
3. Complexity level (beginner/intermediate/advanced)
4. Key intent (what was the user trying to achieve?)
"""

    try:
        response = client.responses.create(model=MODEL, input=input_text)

        return {
            "analysis": response.output_text if hasattr(response, "output_text") else str(response),
            "platform": platform,
            "title": title,
        }
    except Exception as e:
        print(f"   ⚠️  Error analyzing conversation '{title}': {e}")
        return {"analysis": f"Error: {str(e)}", "platform": platform, "title": title}

### `extract_platform_themes`
Runs `analyze_conversation_with_llm` over the top-N conversations for a platform, then asks the LLM to aggregate the results into themes, use-cases, and key insights.

In [ ]:
def extract_platform_themes(client, conversations: pd.DataFrame, platform: str, sample_size: int = 50) -> dict:
    """Extract themes from top conversations per platform"""
    print(f"\n   📊 Analyzing {platform.upper()} conversations...")

    sampled_convs = conversations.head(sample_size)

    all_summaries = []
    for idx, row in sampled_convs.iterrows():
        print(f"      Processing {idx + 1}/{len(sampled_convs)}...", end="\r")
        analysis = analyze_conversation_with_llm(client, row["title"], row["user_queries"], platform)
        all_summaries.append(
            {
                "title": row["title"],
                "message_count": row["message_count"],
                "analysis": analysis["analysis"],
                "date": row["conversation_create_time"],
            }
        )

    print(f"\n      ✅ Analyzed {len(all_summaries)} conversations")

    aggregate_prompt = f"""Based on these {len(all_summaries)} conversation analyses for {platform.upper()},
provide an aggregate summary with:

1. Top 5 themes/topics (with frequency %)
2. Primary use cases (ranked)
3. Complexity distribution (beginner/intermediate/advanced %)
4. Key insights about how this platform is used

Conversation Analyses:
{json.dumps(all_summaries[:30], indent=2)}

Respond in JSON format with keys: themes, use_cases, complexity, insights
"""

    try:
        response = client.responses.create(model=MODEL, input=aggregate_prompt)

        result = response.output_text if hasattr(response, "output_text") else str(response)

        try:
            parsed = json.loads(result)
            return parsed
        except json.JSONDecodeError:
            return {
                "raw_response": result,
                "themes": "See raw_response",
                "use_cases": "See raw_response",
                "complexity": "See raw_response",
                "insights": result,
            }
    except Exception as e:
        print(f"   ⚠️  Error creating aggregate summary: {e}")
        return {"error": str(e)}

### `compare_platforms`
Takes the two platform theme summaries and asks the LLM to highlight differences, strengths, and notable patterns.

In [ ]:
def compare_platforms(client, chatgpt_themes: dict, claude_themes: dict) -> dict:
    """Compare usage patterns across platforms"""
    print("\n   🔍 Comparing platforms...")

    comparison_prompt = f"""Compare ChatGPT vs Claude usage based on these analyses:

ChatGPT Summary:
{json.dumps(chatgpt_themes, indent=2)}

Claude Summary:
{json.dumps(claude_themes, indent=2)}

Provide a comparison highlighting:
1. Unique strengths of each platform
2. Different use case preferences
3. Complexity differences
4. Notable patterns or trends

Respond in JSON format with keys: chatgpt_strengths, claude_strengths, use_case_differences,
complexity_comparison, key_findings
"""

    try:
        response = client.responses.create(model=MODEL, input=comparison_prompt)

        result = response.output_text if hasattr(response, "output_text") else str(response)

        try:
            return json.loads(result)
        except json.JSONDecodeError:
            return {"comparison": result}
    except Exception as e:
        print(f"   ⚠️  Error comparing platforms: {e}")
        return {"error": str(e)}

---
## Step 3: Run the Analysis

Run the cells below in order. Each cell is one stage of the pipeline.

> **Tip:** If you want to re-run just the comparison without re-calling the API, skip straight to the *Compare platforms* cell — `chatgpt_themes` and `claude_themes` are already in memory.

### Load API key
Reads your `.env` file. You'll see an error here if `OPENAI_API_KEY` is not set.

In [ ]:
from dotenv import load_dotenv

load_dotenv()
client = get_openai_client()
print("OpenAI client ready.")

### Load raw data
Reads the JSONL files from `data/staging/`.

In [ ]:
chatgpt_df = load_chats(DATA_DIR / "chatgpt_messages.jsonl")
claude_df = load_chats(DATA_DIR / "claude_messages.jsonl")
print(f"ChatGPT: {len(chatgpt_df):,} messages")
print(f"Claude:  {len(claude_df):,} messages")

### Prepare conversation summaries
Filters to conversations with 3+ messages and sorts by length. The top `sample_size` will be sent to the LLM.

In [ ]:
chatgpt_convs = get_conversation_summaries(chatgpt_df, "chatgpt")
claude_convs = get_conversation_summaries(claude_df, "claude")
print(f"ChatGPT: {len(chatgpt_convs)} qualifying conversations")
print(f"Claude:  {len(claude_convs)} qualifying conversations")

### Analyse ChatGPT conversations
Calls the LLM for each conversation. This may take a minute.

In [ ]:
chatgpt_themes = extract_platform_themes(client, chatgpt_convs, "chatgpt", sample_size=30)

### Analyse Claude conversations
Same process for Claude.

In [ ]:
claude_themes = extract_platform_themes(client, claude_convs, "claude", sample_size=30)

### Compare platforms
Asks the LLM to compare usage patterns between the two platforms.

In [ ]:
comparison = compare_platforms(client, chatgpt_themes, claude_themes)

### Review results
Print the full analysis. You can also inspect individual keys like `chatgpt_themes['themes']`.

In [ ]:

print("=== ChatGPT ===")
print(json.dumps(chatgpt_themes, indent=2))
print("\n=== Claude ===")
print(json.dumps(claude_themes, indent=2))
print("\n=== Comparison ===")
print(json.dumps(comparison, indent=2))

### Save results
Writes the full analysis to `data/figures/llm_conversation_analysis.json`.

In [ ]:
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = {
    "chatgpt_analysis": chatgpt_themes,
    "claude_analysis": claude_themes,
    "platform_comparison": comparison,
    "metadata": {
        "model": MODEL,
        "chatgpt_conversations_analyzed": len(chatgpt_convs),
        "claude_conversations_analyzed": len(claude_convs),
    },
}
out_file = OUTPUT_DIR / "llm_conversation_analysis.json"
out_file.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"Saved to {out_file}")